# ThreatIntel-Agent Proposal Demo: Neon DB + Gradio UI

이 노트북은 GitHub에서 프로젝트 코드를 clone하고, 이미 배포된 Neon PostgreSQL `threat_intel_agent` DB에 접속해 과제 산출물을 검증하는 Colab 데모입니다.

구성:
- GitHub repo clone
- Neon DB 테이블 row count 확인
- 제안서 샘플 SQL 10개 실행 검증
- OpenAI `gpt-4o-mini` 기반 자연어 → SQL → 분석 실행
- Gradio UI 실행


## 4단계 마일스톤 대응

| 단계 | 제출물 | 이 노트북/프로젝트의 대응 |
|---|---|---|
| 과제 #1 | 프로젝트 제안서 Markdown | 스키마, ERD, 질문 10개, 기대 SQL, 데이터 샘플 10행 반영 |
| 과제 #2 | Neon PostgreSQL 배포 + DSN 공유 | `threat_intel_agent` database에 시드 데이터 적재 완료, DSN 입력 후 row count 확인 |
| 과제 #3 | Colab 노트북 + LangSmith trace URL | SQL 10개 검증, OpenAI LLM 자연어 질의, Gradio UI 실행 |
| 최종 발표 | 라이브 데모 + 발표 | Gradio UI에서 자연어 질의와 직접 SQL 실행 시연 |


## 1. GitHub 프로젝트 가져오기

아래 셀은 `jazdr/threat-analysis-agent` repo를 Colab 런타임으로 가져옵니다. 이미 clone되어 있으면 재사용합니다.


In [ ]:
%%bash
if [ -d threat-analysis-agent ]; then
  git -C threat-analysis-agent pull --ff-only
else
  git clone https://github.com/jazdr/threat-analysis-agent.git
fi


In [ ]:
%cd threat-analysis-agent
!pwd
!git log --oneline -1
!ls -la colab_support notebooks | sed -n '1,80p' 


## 2. 의존성 설치

Colab 런타임 기준으로 필요한 패키지를 설치합니다.


In [ ]:
!pip -q install psycopg2-binary pandas openai gradio python-dotenv


## 3. Neon / OpenAI 환경변수 설정

Neon 콘솔 또는 제출 안내에서 받은 접속 문자열을 입력합니다. 기본 `neondb` DSN을 넣어도 아래 셀이 자동으로 `threat_intel_agent` database를 가리키도록 변환합니다.

OpenAI API Key는 자연어 → SQL 생성, 결과 분석, Gradio 질문 실행에 사용됩니다. 제출용 노트북에는 실제 키를 저장하지 않고 실행 시점에 직접 입력합니다.


In [ ]:
import os
import sys
from pathlib import Path
from getpass import getpass

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

neon_database_url = getpass('Neon DATABASE_URL, 기본 neondb 또는 threat_intel_agent DSN: ').strip()
openai_api_key = getpass('OpenAI API Key, sk-... : ').strip()

if not neon_database_url:
    raise ValueError('Neon DATABASE_URL을 입력해야 합니다.')
if not openai_api_key:
    raise ValueError('OpenAI API Key를 입력해야 LLM 질의와 Gradio 질문 실행이 가능합니다.')

from colab_support.neon_gradio_threat_agent import (
    ColabThreatIntelAgent,
    build_gradio_app,
    database_url_for_name,
    table_counts,
)

TARGET_DATABASE = 'threat_intel_agent'
os.environ['DATABASE_URL'] = database_url_for_name(neon_database_url, TARGET_DATABASE)
os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'

# OpenAI-compatible endpoint를 쓰는 경우만 설정하세요. 기본 OpenAI API 사용 시 비워둡니다.
# os.environ['OPENAI_BASE_URL'] = 'https://api.openai.com/v1'

print('colab_support import OK')
print('Using Neon database:', TARGET_DATABASE)
print('DATABASE_URL host/db:', os.environ['DATABASE_URL'].split('@')[-1])
print('OpenAI model:', os.environ['OPENAI_MODEL'])
print('OpenAI API key loaded:', bool(os.environ.get('OPENAI_API_KEY')))


## 4. Neon DB 테이블 현황 확인

과제 #2 제출물인 Neon PostgreSQL 배포 상태를 확인합니다. 정상 데이터 기준 row count는 `otx_threat_intel=2340`, `cve_vulnerabilities=1585`, `malicious_domains=162`, `malicious_ips=200`, `cisa_known_exploited_vulnerabilities=1587`, `threat_intel_links=500`입니다.


In [ ]:
counts = table_counts()
display(counts)

required_tables = {
    'otx_threat_intel',
    'cve_vulnerabilities',
    'malicious_domains',
    'malicious_ips',
    'cisa_known_exploited_vulnerabilities',
    'threat_intel_links',
}
existing_tables = set(counts['table'].tolist()) if 'table' in counts.columns else set()
missing_tables = sorted(required_tables - existing_tables)
if missing_tables:
    raise RuntimeError(
        '필수 테이블이 없습니다. 입력한 DSN이 threat_intel_agent DB를 가리키는지 확인하세요. '
        f'missing={missing_tables}, current_url={os.environ["DATABASE_URL"].split("@")[ -1 ]}'
    )
print('Required tables OK')


## 5. 제안서 샘플 SQL 10개 실행 검증

아래 셀은 과제 #1 제안서의 기본 10개 기대 SQL이 실제 Neon DB에서 실행되는지 확인합니다. 현재 적재 데이터 기준 예상 행 수는 Q1=1, Q2=2, Q3=5, Q4=17, Q5=0, Q6=5, Q7=10, Q8=134, Q9=10, Q10=5입니다.


In [ ]:
agent = ColabThreatIntelAgent(api_key=os.environ['OPENAI_API_KEY'])

EXPECTED_SQL = {
    'Q1': "SELECT COUNT(*) AS high_risk_ip_count FROM malicious_ips WHERE threat_severity = 'High';",
    'Q2': "SELECT cve_id, vendor_project, product, vulnerability_name FROM cve_vulnerabilities WHERE date_added = '2026-04-28';",
    'Q3': "SELECT domain, reputation, threat_severity, malicious_votes FROM malicious_domains WHERE data_source = 'VirusTotal' ORDER BY reputation ASC LIMIT 5;",
    'Q4': "SELECT country, COUNT(*) AS malicious_ip_count, ROUND(AVG(reputation_score), 2) AS avg_reputation FROM malicious_ips WHERE malicious_votes > 0 GROUP BY country ORDER BY malicious_ip_count DESC;",
    'Q5': "WITH phishing_pulses AS (SELECT pulse_id, title, tags, attack_ids FROM otx_threat_intel WHERE LOWER(tags) LIKE '%phishing%') SELECT DISTINCT c.cve_id, c.vendor_project, c.product, p.title FROM cve_vulnerabilities c JOIN phishing_pulses p ON LOWER(c.short_description) LIKE '%phish%' ORDER BY c.cve_id;",
    'Q6': "SELECT cve_id, vendor_project, product, due_date, short_description FROM cve_vulnerabilities WHERE known_ransomware_campaign_use = 'Known' OR LOWER(short_description) LIKE '%ransomware%' ORDER BY due_date ASC LIMIT 5;",
    'Q7': "SELECT ip, owner, network, country, malicious_votes, threat_severity FROM malicious_ips WHERE tor_node = 'Yes' AND malicious_votes >= 2 ORDER BY malicious_votes DESC;",
    'Q8': "WITH network_stats AS (SELECT network, AVG(reputation_score) AS avg_reputation FROM malicious_ips GROUP BY network), ranked_ips AS (SELECT ip, network, reputation_score, ROW_NUMBER() OVER (PARTITION BY network ORDER BY reputation_score ASC) AS rn FROM malicious_ips WHERE reputation_score IS NOT NULL) SELECT r.ip, r.network, r.reputation_score, ns.avg_reputation FROM ranked_ips r JOIN network_stats ns ON r.network = ns.network WHERE r.rn <= 3 ORDER BY r.network, r.rn;",
    'Q9': "WITH lumma_base AS (SELECT pulse_id, created, attack_ids FROM otx_threat_intel WHERE LOWER(tags || ' ' || COALESCE(malware_families,'')) LIKE '%lumma%' AND attack_ids IS NOT NULL), attack_list AS (SELECT DISTINCT TRIM(t.aid) AS attack_id FROM lumma_base lb, LATERAL (SELECT unnest(string_to_array(lb.attack_ids, ',')) AS aid) t WHERE TRIM(t.aid) <> '') SELECT DISTINCT o.pulse_id, o.title, o.created, o.tags, o.attack_ids FROM otx_threat_intel o JOIN attack_list a ON ',' || REPLACE(o.attack_ids, ' ', '') || ',' LIKE '%,' || a.attack_id || ',%' WHERE o.pulse_id NOT IN (SELECT pulse_id FROM lumma_base) ORDER BY o.created DESC LIMIT 10;",
    'Q10': "WITH tag_list AS (SELECT TRIM(t.tag) AS tag, o.created FROM otx_threat_intel o, LATERAL (SELECT unnest(string_to_array(o.tags, ',')) AS tag) t WHERE o.created >= DATE '2026-04-03' AND o.tags IS NOT NULL AND TRIM(t.tag) <> ''), tag_counts AS (SELECT tag, COUNT(*) AS tag_count, MIN(created) AS first_seen, MAX(created) AS last_seen FROM tag_list GROUP BY tag HAVING COUNT(*) >= 2) SELECT tag, tag_count, first_seen, last_seen, ROW_NUMBER() OVER (ORDER BY tag_count DESC, last_seen DESC) AS rank FROM tag_counts ORDER BY tag_count DESC, last_seen DESC LIMIT 5;",
}
EXPECTED_ROWS = {'Q1': 1, 'Q2': 2, 'Q3': 5, 'Q4': 17, 'Q5': 0, 'Q6': 5, 'Q7': 10, 'Q8': 134, 'Q9': 10, 'Q10': 5}

for qid, sql in EXPECTED_SQL.items():
    rows, _ = agent.execute_sql(sql, limit=1000)
    status = 'OK' if len(rows) == EXPECTED_ROWS[qid] else 'CHECK'
    print(f'{qid}: {status}, rows={len(rows)}, expected={EXPECTED_ROWS[qid]}')


## 5-1. 추가 상관분석 SQL 5개 실행 검증

아래 셀은 기본 10개 질문 외에 추가한 `(상관분석)` 조인 질문 5개를 검증합니다.


In [ ]:
CORRELATION_SQL = {
    'A1': "SELECT o.pulse_id, o.title, o.tags, c.cve_id, c.vendor_project, c.product, l.relation_type, l.confidence FROM threat_intel_links l JOIN otx_threat_intel o ON l.pulse_id = o.pulse_id JOIN cve_vulnerabilities c ON l.cve_id = c.cve_id ORDER BY l.confidence DESC, o.created DESC LIMIT 10;",
    'A2': "SELECT c.cve_id, c.vendor_project, c.product, c.due_date, o.pulse_id, o.title, o.tags, l.confidence FROM threat_intel_links l JOIN cve_vulnerabilities c ON l.cve_id = c.cve_id JOIN otx_threat_intel o ON l.pulse_id = o.pulse_id WHERE c.known_ransomware_campaign_use = 'Known' OR LOWER(o.tags || ' ' || COALESCE(o.title,'')) LIKE '%ransomware%' ORDER BY c.due_date ASC, l.confidence DESC LIMIT 10;",
    'A3': "SELECT c.product, COUNT(DISTINCT c.cve_id) AS cve_count, COUNT(DISTINCT o.pulse_id) AS otx_report_count, ROUND(AVG(l.confidence), 2) AS avg_confidence FROM threat_intel_links l JOIN cve_vulnerabilities c ON l.cve_id = c.cve_id JOIN otx_threat_intel o ON l.pulse_id = o.pulse_id WHERE c.product IN ('Android', 'Windows', 'Pixel', 'Defender') GROUP BY c.product ORDER BY cve_count DESC, otx_report_count DESC;",
    'A4': "SELECT o.attack_ids, c.cve_id, c.vendor_project, c.product, o.title, l.confidence FROM threat_intel_links l JOIN otx_threat_intel o ON l.pulse_id = o.pulse_id JOIN cve_vulnerabilities c ON l.cve_id = c.cve_id WHERE o.attack_ids IS NOT NULL AND o.attack_ids <> 'Unknown' ORDER BY l.confidence DESC, o.created DESC LIMIT 10;",
    'A5': "SELECT o.created, o.title, c.cve_id, c.vendor_project, c.product, c.due_date, c.known_ransomware_campaign_use, l.confidence FROM threat_intel_links l JOIN otx_threat_intel o ON l.pulse_id = o.pulse_id JOIN cve_vulnerabilities c ON l.cve_id = c.cve_id WHERE c.due_date IS NOT NULL ORDER BY c.due_date ASC, o.created DESC LIMIT 10;",
}
CORRELATION_ROWS = {'A1': 10, 'A2': 10, 'A3': 3, 'A4': 10, 'A5': 10}

for qid, sql in CORRELATION_SQL.items():
    rows, _ = agent.execute_sql(sql, limit=1000)
    status = 'OK' if len(rows) == CORRELATION_ROWS[qid] else 'CHECK'
    print(f'{qid}: {status}, rows={len(rows)}, expected={CORRELATION_ROWS[qid]}')


## 6. 자연어 질의 Smoke Test

자연어 → SQL → 실행 → 분석 흐름이 정상인지 확인합니다. 이 셀부터는 입력한 OpenAI API Key로 `gpt-4o-mini`를 호출합니다.


In [ ]:
agent = ColabThreatIntelAgent(api_key=os.environ['OPENAI_API_KEY'], model=os.environ.get('OPENAI_MODEL', 'gpt-4o-mini'))
result = agent.ask('(상관분석) OTX 보고서와 CVE 취약점이 연결된 사례를 조인해서 보여줘')
print('ERROR:', result.error)
print('SQL:\n', result.sql)
print('ROWS:', len(result.rows))
print(result.analysis[:1000])


## 7. Gradio UI 실행

Colab에서는 `share=True`로 실행하면 외부 접속 가능한 URL이 생성됩니다.


In [ ]:
demo = build_gradio_app(agent)
demo.launch(share=True, debug=True)


## 8. 데모 질문 예시

기본 질문:
- 심각도가 High인 악성 IP가 몇 개인가요?
- 2026년 4월 28일에 CISA KEV에 추가된 취약점 목록을 보여주세요.
- 데이터 소스가 VirusTotal인 도메인 중 평판 점수가 가장 낮은 TOP 5를 알려주세요.
- 국가별로 악성으로 판정받은 IP 개수를 집계해주세요.
- phishing 키워드를 포함한 OTX 보고서와 연관된 CVE가 있나요?
- 랜섬웨어로 알려진 취약점 중 대응 마감일이 가장 임박한 5개를 보여주세요.
- Tor 노드로 확인된 IP들 중 악성 판정이 2개 이상인 주소와 소유 조직을 보여주세요.
- OTX 보고서에서 Lumma 관련 태그와 동일 ATT&CK ID를 공유하는 보고서를 보여주세요.
- CISA KEV 원본 카탈로그에서 최근 추가된 취약점 5개를 보여줘.

추가 상관분석 질문:
- (상관분석) OTX 보고서와 CVE 취약점이 연결된 사례를 조인해서 보여주세요.
- (상관분석) 랜섬웨어 관련 CVE와 연결된 OTX 보고서를 보여주세요.
- (상관분석) Android/Windows 제품 취약점과 관련된 OTX 위협 보고서 수를 비교해주세요.
- (상관분석) ATT&CK ID가 있는 OTX 보고서와 연결된 CVE를 신뢰도 순으로 보여주세요.
- (상관분석) 최근 OTX 보고서와 대응 마감일이 임박한 CVE를 함께 보여주세요.
